In [6]:
import numpy as np
from tqdm import tqdm

def create_starting_set(r, n):
    multiplier = np.zeros(n)
    q_n = int((1 + r**n * (r - 2)) / (r - 1))
    set = np.zeros([q_n, n])
    for j in range(n):
        multiplier[j] = r**j
    for i in range(q_n):
        set[i] = (i * multiplier) % (q_n)
    return set

def expand_set(set, q):
    # For each original vertex:
    # - coordinates >= q are always shifted by +q in every copy
    # - coordinates < q are duplicated with both original and +q values
    #   across all combinations
    dim = set.shape[1]
    expanded_vertices = []

    for v in set:
        low_indices = []
        base_v = np.copy(v)

        for j in range(dim):
            if v[j] < q:
                low_indices.append(j)
            else:
                base_v[j] = v[j] + q

        for mask in range(2 ** len(low_indices)):
            new_v = np.copy(base_v)
            for bit, idx in enumerate(low_indices):
                if (mask >> bit) & 1:
                    new_v[idx] = v[idx] + q
                else:
                    new_v[idx] = v[idx]
            expanded_vertices.append(new_v)

    return np.array(expanded_vertices)

def make_directed_graph(set, m, p, q, new_p, new_q, prev_p, prev_q):
    num_vertices = set.shape[0]
    num_coords = set.shape[1]
    graph = np.zeros([num_vertices, num_vertices])
    for i in range(num_vertices):
        for j in range(num_vertices):
            if set[i][m] == (set[j][m] - new_q) % new_p:
                closes = True
                for k in range(m):
                    x = (set[i][k] - set[j][k]) % prev_p[k]
                    if x >= prev_q[k] and x <= prev_p[k] - prev_q[k]:
                        closes = False
                        break
                if closes:
                    for k in range(m + 1, num_coords):
                        x = (set[i][k] - set[j][k]) % p
                        if x >= q and x <= p - q:
                            closes = False
                            break
                if closes:
                    graph[i][j] = 1
                
    return graph

def check_cycle(directed_graph):
    # Check if the directed graph contains a cycle using Depth-First Search (DFS)
    num_vertices = directed_graph.shape[0]
    visited = [0] * num_vertices  # 0 = unvisited, 1 = visiting, 2 = visited

    def dfs(vertex):
        if visited[vertex] == 1:
            return True
        if visited[vertex] == 2:
            return False

        visited[vertex] = 1
        for neighbor in range(num_vertices):
            if directed_graph[vertex][neighbor] != 0 and dfs(neighbor):
                return True
        visited[vertex] = 2
        return False

    for vertex in range(num_vertices):
        if visited[vertex] == 0 and dfs(vertex):
            return True

    return False

def largest_directed_path(directed_graph, vertex):
    # Returns the length of the longest directed path ending at vertex.
    # Length is measured in edges, so a single edge has length 1.
    num_vertices = directed_graph.shape[0]
    memo = [-1] * num_vertices
    visiting = [False] * num_vertices

    def longest_path_to(current_vertex):
        if memo[current_vertex] != -1:
            return memo[current_vertex]
        if visiting[current_vertex]:
            raise ValueError("The graph contains a directed cycle, so the longest path is undefined.")

        visiting[current_vertex] = True
        best_length = 0

        for predecessor in range(num_vertices):
            if directed_graph[predecessor][current_vertex] != 0:
                candidate_length = longest_path_to(predecessor) + 1
                if candidate_length > best_length:
                    best_length = candidate_length

        visiting[current_vertex] = False
        memo[current_vertex] = best_length
        return best_length

    return longest_path_to(vertex)

def shift_set(set, directed_graph, m, p):
    largest = np.zeros(set.shape[0])
    max_path = 0
    for i in range(set.shape[0]):
        largest[i] = largest_directed_path(directed_graph, i)
    max_path = max(largest)
    for i in range(set.shape[0]):
        set[i][m] = (set[i][m] * (max_path + 1) + largest[i]) % (p * (max_path + 1))
    return set, max_path

def number_distinct_values(set,m):
    distinct_v = []
    #List all distinct values in the m-th coordinate of the set in increasing order
    for i in range(set.shape[0]):
        value = int(set[i][m])
        if value not in distinct_v:
            distinct_v.append(value)
    distinct_v.sort()
    return distinct_v

def create_forward_adjacency(set, p, q):
    count = np.zeros(len(set))
    for i in range(len(set)):
        for k in range(len(set)):
            diff = (set[k] - set[i]) % p
            if diff < q:
                count[i] += 1
    return count-1

def compress_set(set, m, p, q):
    # Keep deleting the first vertex in the first pair whose forward adjacencies differ by 1
    # until all forward adjacencies are equal.
    distinct_v = number_distinct_values(set, m)
    # print(distinct_v, len(distinct_v))
    # print(create_forward_adjacency(distinct_v, p, q))
    indices = np.zeros(set.shape[0])
    for i in range(set.shape[0]):
        indices[i] = distinct_v.index(set[i][m])
    while True:
        adjacency = create_forward_adjacency(distinct_v, p, q)
        # print(adjacency)
        if np.all(adjacency == adjacency[0]):
            for i in range(set.shape[0]):
                set[i][m] = indices[i]
            return set, len(distinct_v), adjacency[0]+1

        delete_index = None
        for i in range(len(distinct_v) - 1):
            if adjacency[i + 1] == adjacency[i] - 1:
                delete_index = i
                for j in range(set.shape[0]):
                    if indices[j] > delete_index:
                        indices[j] -= 1
                break

        if delete_index is None:
            delete_index = len(distinct_v) - 1
            for i in range(len(indices)):
                if indices[i] == delete_index:
                    indices[i] = 0

        distinct_v = np.delete(distinct_v, delete_index, axis=0)


r = 3
n = 3
S = create_starting_set(r, n)
q_n1 = int((1 + (r) ** (n - 1) * (r - 2)) / (r - 1))
q_n = int((1 + r**n * (r - 2)) / (r - 1))
p = q_n + q_n1
q = q_n1
new_p = p
new_q = q


# two_counts = np.zeros(n+1)
# for i in range(len(S)):
#     count = 0
#     for j in range(n):
#         if S[i][j] < q:
#             count += 1
#     two_counts[count] += 1
#     print(2**count)
# print(two_counts)

prev_p = []
prev_q = []
S = expand_set(S, new_q)
print(S)
import sys
np.set_printoptions(threshold=sys.maxsize)
for i in tqdm(range(S.shape[1])):
    new_p = p
    new_q = q
    A = make_directed_graph(S, i, p, q, new_p, new_q, prev_p, prev_q)
    while not check_cycle(A):
        #Count 1's in A
        S, max_path = shift_set(S, A, i, new_p)
        new_p = (max_path + 1) * new_p
        new_q = (max_path + 1) * new_q + 1
        S, new_p, new_q = compress_set(S, i, new_p, new_q)
        print(new_p, new_q, i)
        A = make_directed_graph(S, i, p, q, new_p, new_q, prev_p, prev_q)
    prev_p.append(new_p)
    prev_q.append(new_q)

# print(S, new_p, new_q, prev_p, prev_q)
# i = 0
# prev_q = new_q
# prev_p = new_p
# A = make_directed_graph(S, i, p, q, new_p, new_q, p, q)
# print(new_p, new_q, new_p/new_q)
# steps = 0
# while not check_cycle(A):
#     #Count 1's in A
#     steps += 1
#     S, max_path = shift_set(S, A, i, new_p)
#     # print(S, max_path)
#     new_p = (max_path + 1) * new_p
#     new_q = (max_path + 1) * new_q + 1
#     S, new_p, new_q = compress_set(S, i, new_p, new_q)
#     # print(new_p, new_q, new_p/new_q, prev_p/prev_q-new_p/new_q)
#     prev_p = new_p
#     prev_q = new_q
#     A = make_directed_graph(S, i, p, q, new_p, new_q, p, q)
#Sort on the first coordinate

print(S[S[:, 0].argsort()], new_p, new_q)

[[ 0.  0.  0.]
 [ 5.  0.  0.]
 [ 0.  5.  0.]
 [ 5.  5.  0.]
 [ 0.  0.  5.]
 [ 5.  0.  5.]
 [ 0.  5.  5.]
 [ 5.  5.  5.]
 [ 1.  3. 14.]
 [ 6.  3. 14.]
 [ 1.  8. 14.]
 [ 6.  8. 14.]
 [ 2. 11.  4.]
 [ 7. 11.  4.]
 [ 2. 11.  9.]
 [ 7. 11.  9.]
 [ 3. 14. 18.]
 [ 8. 14. 18.]
 [ 4. 17. 13.]
 [ 9. 17. 13.]
 [10.  1.  3.]
 [10.  6.  3.]
 [10.  1.  8.]
 [10.  6.  8.]
 [11.  4. 17.]
 [11.  9. 17.]
 [12. 12. 12.]
 [13. 15.  2.]
 [13. 15.  7.]
 [14. 18. 16.]
 [15.  2. 11.]
 [15.  7. 11.]
 [16. 10.  1.]
 [16. 10.  6.]
 [17. 13. 15.]
 [18. 16. 10.]]


 33%|███▎      | 1/3 [00:00<00:00,  5.55it/s]

26 7.0 0
22 6.0 0
29 8.0 0
36 10.0 0
26 7.0 1


100%|██████████| 3/3 [00:00<00:00,  5.73it/s]

22 6.0 1
29 8.0 1
36 10.0 1
26 7.0 2
22 6.0 2
29 8.0 2
36 10.0 2
[[ 0. 10. 13.]
 [ 1. 13.  3.]
 [ 2. 31. 20.]
 [ 3.  0. 10.]
 [ 4. 18. 29.]
 [ 5. 21. 19.]
 [ 6.  3.  0.]
 [ 7.  8. 26.]
 [ 8. 26.  9.]
 [ 9. 29. 35.]
 [10. 11. 16.]
 [11. 16.  6.]
 [12. 34. 25.]
 [13.  1. 15.]
 [14. 19. 32.]
 [15. 24. 22.]
 [16.  6.  5.]
 [17.  9. 31.]
 [18. 27. 12.]
 [19. 32.  2.]
 [20. 14. 21.]
 [21. 17. 11.]
 [22. 35. 28.]
 [23.  4. 18.]
 [24. 22.  1.]
 [25. 25. 27.]
 [26.  7.  8.]
 [27. 12. 34.]
 [28. 30. 17.]
 [29. 33.  7.]
 [30. 15. 24.]
 [31. 20. 14.]
 [32.  2. 33.]
 [33.  5. 23.]
 [34. 23.  4.]
 [35. 28. 30.]] 36 10.0


In [12]:
def check_independent(set, p, q):
    for i in range(set.shape[0]):
        for j in range(i+1,set.shape[0]):
            X = False
            for k in range(set.shape[1]):
                diff = (set[i][k]-set[j][k])%p[k]
                if diff >= q[k] and diff <= p[k]-q[k]:
                    X = True
            if not X:
                return False
    return True
print(check_independent(S, [new_p,new_p,new_p], [new_q,new_q,new_q]))

True


In [6]:
### VERSION 2, TRYING TO IMPROVE THE SHIFTING PROCESS
def check_space(set, m, p, q, new_p, new_q, prev_p, prev_q):
    #Check for every vertex, how much space is left (most will be 0 but if there is no cycle at least one has a value larger than 0)
    #In the original algorithm, we always assume it is 1, but in some cases it can be larger, which allows us to shift by more than 1 and speed up the process
    space = new_p*np.ones(set.shape[0])
    num_vertices = set.shape[0]
    num_coords = set.shape[1]
    for i in range(num_vertices):
        for j in range(num_vertices):
            closes = True
            for k in range(m):
                x = (set[i][k] - set[j][k]) % prev_p
                if x >= prev_q and x <= prev_p - prev_q:
                    closes = False
                    break
            if closes:
                for k in range(m + 1, num_coords):
                    x = (set[i][k] - set[j][k]) % p
                    if x >= q and x <= p - q:
                        closes = False
                        break
            if closes:
                diff = (set[j][m] - new_q - set[i][m]) % new_p
                if diff < space[i]:
                    space[i] = diff

    return space

def shift_set_version2(set, directed_graph, m, p):
    largest = np.zeros(set.shape[0])
    max_path = 0
    for i in range(set.shape[0]):
        largest[i] = largest_directed_path(directed_graph, i)
    max_path = max(largest)
    space = check_space(set, m, p, q, new_p, new_q, prev_p, prev_q)
    # Available space is the space at the index for wich largest is maximum
    available_space = space[largest.argmax()]
    for i in range(set.shape[0]):
        set[i][m] = (set[i][m] * (max_path + 1) + available_space*largest[i]) % (p * (max_path + 1))
    return set, max_path, available_space

def shift_set_version3(set, directed_graph, m, p):
    largest = np.zeros(set.shape[0])
    max_path = 0
    for i in range(set.shape[0]):
        largest[i] = largest_directed_path(directed_graph, i)
    max_path = max(largest)
    space = check_space(set, m, p, q, new_p, new_q, prev_p, prev_q)
    # Check all vertices for which the directed graph is all zeros, and take the sum of their spaces as available space, since we can shift all of them at the same time
    available_space = 0
    for i in range(set.shape[0]):
        if np.all(directed_graph[i] == 0):
            available_space += space[i]
    for i in range(set.shape[0]):
        set[i][m] = (set[i][m] * (max_path + 1) + available_space*largest[i]) % (p * (max_path + 1))
    return set, max_path, available_space

r = 4
n = 3
S = create_starting_set(r, n)
q_n1 = int((1 + (r) ** (n - 1) * (r - 2)) / (r - 1))
q_n = int((1 + r**n * (r - 2)) / (r - 1))
p = q_n + q_n1
q = q_n1
new_p = p
new_q = q

S = expand_set(S, new_q)
print(S)
import sys
np.set_printoptions(threshold=sys.maxsize)

i = 0
prev_q = new_q
prev_p = new_p
A = make_directed_graph(S, i, p, q, new_p, new_q, p, q)
print(new_p, new_q, new_p/new_q)
steps = 0
while not check_cycle(A):
    steps += 1
    #Count 1's in A
    S, max_path, available_space = shift_set_version2(S, A, i, new_p)
    print(S, max_path, available_space)
    new_p = (max_path + 1) * new_p
    new_q = (max_path + 1) * new_q + available_space
    S, new_p, new_q = compress_set(S, i, new_p, new_q)
    print(new_p, new_q, new_p/new_q, prev_p/prev_q-new_p/new_q)
    prev_p = new_p
    prev_q = new_q
    A = make_directed_graph(S, i, p, q, new_p, new_q, p, q)
print(new_p, new_q)
print(S)
print(f"finished in {steps} steps")

[[ 0.  0.  0.]
 [11.  0.  0.]
 [ 0. 11.  0.]
 [11. 11.  0.]
 [ 0.  0. 11.]
 [11.  0. 11.]
 [ 0. 11. 11.]
 [11. 11. 11.]
 [ 1.  4. 27.]
 [12.  4. 27.]
 [ 1. 15. 27.]
 [12. 15. 27.]
 [ 2.  8. 43.]
 [13.  8. 43.]
 [ 2. 19. 43.]
 [13. 19. 43.]
 [ 3. 23.  5.]
 [14. 23.  5.]
 [ 3. 23. 16.]
 [14. 23. 16.]
 [ 4. 27. 32.]
 [15. 27. 32.]
 [ 5. 31. 48.]
 [16. 31. 48.]
 [ 6. 35. 10.]
 [17. 35. 10.]
 [ 6. 35. 21.]
 [17. 35. 21.]
 [ 7. 39. 37.]
 [18. 39. 37.]
 [ 8. 43. 53.]
 [19. 43. 53.]
 [ 9. 47. 26.]
 [20. 47. 26.]
 [10. 51. 42.]
 [21. 51. 42.]
 [22.  1.  4.]
 [22. 12.  4.]
 [22.  1. 15.]
 [22. 12. 15.]
 [23.  5. 31.]
 [23. 16. 31.]
 [24.  9. 47.]
 [24. 20. 47.]
 [25. 24.  9.]
 [25. 24. 20.]
 [26. 28. 36.]
 [27. 32. 52.]
 [28. 36. 25.]
 [29. 40. 41.]
 [30. 44.  3.]
 [30. 44. 14.]
 [31. 48. 30.]
 [32. 52. 46.]
 [33.  2.  8.]
 [33. 13.  8.]
 [33.  2. 19.]
 [33. 13. 19.]
 [34.  6. 35.]
 [34. 17. 35.]
 [35. 10. 51.]
 [35. 21. 51.]
 [36. 25. 24.]
 [37. 29. 40.]
 [38. 33.  2.]
 [38. 33. 13.]
 [39. 37. 

KeyboardInterrupt: 

In [ ]:

#A directed graph where we for every vertex, show all the ones that are closest, not just 0 distance only.
def make_directed_graph_version2(set, m, p, q, new_p, new_q):
    num_vertices = set.shape[0]
    num_coords = set.shape[1]
    graph = -np.ones([num_vertices, num_vertices])
    for i in range(num_vertices):
        k = 0
        adjusted = 0
        while adjusted == 0:
            for j in range(num_vertices):
                if set[i][m] == (set[j][m] + k - new_q) % new_p:
                    closes = True
                    for l in range(m + 1, num_coords):
                        x = (set[i][l] - set[j][l]) % p
                        if x >= q and x <= p - q:
                            closes = False
                            break
                    if closes:
                        graph[i][j] = k
                        adjusted = 1
            k += 1               
    return graph 


4.5
4.5


In [10]:
def rearrange_set(set):
    # Rearrange the set so that the 1st coordinate is in increasing order, and the rest of the coordinates are rearranged accordingly
    indices = np.argsort(set[:, 0])
    return set[indices]

def map_combinations(set):
    #List all combinations of indices, where we name the first element 0, second 1 etc, but for every elements that have the same first value, use all combinations (so in case of 3 elementswith last 2 the same, you would have for example 0,1,2 and 0,2,1 returned)
    combinations = []
    i = 0
    while i < set.shape[0]:
        same_value_indices = [i]
        for j in range(i + 1, set.shape[0]):
            if set[j][0] == set[i][0]:
                same_value_indices.append(j)
            else:
                break
        # Now we have all indices with the same first value, we need to add all combinations of these indices to the list of combinations
        from itertools import permutations
        for perm in permutations(same_value_indices):
            combinations.append(perm)
        i += len(same_value_indices)
    return combinations

#This one has to return [[0,1,2],[1,0,2]]
map_combinations(np.array([[1, 2], [1, 3], [2, 4]]))

[(0, 1), (1, 0), (2,)]

In [2]:
def estimated_guess(set, m, p, q, new_p, new_q):
    num_vertices = set.shape[0]
    num_coords = set.shape[1]
    graph = -np.ones([num_vertices, num_vertices])
    for i in range(num_vertices):
        for j in range(num_vertices):
            if set[i][m] == (set[j][m] - new_q) % new_p:
                closes = True
                for l in range(m + 1, num_coords):
                    x = (set[i][l] - set[j][l]) % p
                    if x >= q and x <= p - q:
                        closes = False
                        break
                if closes:
                    #Count the numbber of values in the set for which at coordinate m, the value is between set[i][m] and set[j][m]
                    count = 0
                    for k in range(num_vertices):
                        if set[i][m] < set[j][m]:
                            if set[k][m] >= set[i][m] and set[k][m] < set[j][m]:
                                count += 1
                        else:
                            if set[k][m] >= set[i][m] or set[k][m] < set[j][m]:
                                count += 1
                    graph[i][j] = count
    return graph, np.max(graph)

def estimated_guess2(set, m, q):
    num_vertices = set.shape[0]
    count = 0
    for i in range(num_vertices):
        if set[i][m] < q:
            count +=1
    return count

r = 3
n = 3
S = create_starting_set(r, n)
print("First job done")
q_n1 = int((1 + (r) ** (n - 1) * (r - 2)) / (r - 1))
q_n = int((1 + r**n * (r - 2)) / (r - 1))
p = q_n + q_n1
q = q_n1

S = expand_set(S, q)
print("Second job done")
A, q_guess = estimated_guess(S, 0, p, q, p, q)
p_guess = len(S)
print(f"Estimated for r:{r}, n:{n} p: {p_guess}, Estimated q: {q_guess}")
sum = 0
count = 0
for i in range(A.shape[0]):
    if np.max(A[i]) != -1:
        sum += np.max(A[i])
        count += 1
print(f"Average number of vertices between two vertices that are connected in the directed graph: {sum / count}")

First job done
Second job done
Estimated for r:3, n:3 p: 36, Estimated q: 10.0
Average number of vertices between two vertices that are connected in the directed graph: 9.67741935483871


In [6]:
def estimated_guess_copilotrework(points, m, p, q, new_p, new_q):
    points = np.asarray(points)
    num_vertices, num_coords = points.shape

    column_m = points[:, m]
    sorted_column_m = np.sort(column_m)
    suffix = points[:, m + 1 :]

    candidate_indices = {}
    for value in np.unique(column_m):
        candidate_indices[value] = np.flatnonzero(column_m == value)

    max_count = -1

    for i in range(num_vertices):
        target_value = (column_m[i] + new_q) % new_p
        js = candidate_indices.get(target_value)
        if js is None or js.size == 0:
            continue

        if suffix.size:
            diff = (suffix[i] - suffix[js]) % p
            keep = ~np.any((diff >= q) & (diff <= p - q), axis=1)
            js = js[keep]
            if js.size == 0:
                continue

        ci = column_m[i]
        cj = column_m[js]
        left = np.searchsorted(sorted_column_m, ci, side="left")
        right = np.searchsorted(sorted_column_m, cj, side="left")
        counts = np.where(ci < cj, right - left, (num_vertices - left) + right)
        current_max = counts.max()
        if current_max > max_count:
            max_count = current_max

    return max_count

r = 4
n = 5

for r in range(3, 8):
    for n in range(1, 8):
        S = create_starting_set(r, n)
        q_n1 = int((1 + (r) ** (n - 1) * (r - 2)) / (r - 1))
        q_n = int((1 + r**n * (r - 2)) / (r - 1))
        p = q_n + q_n1
        q = q_n1

        S = expand_set(S, q)
        #q_guess = estimated_guess_copilotrework(S, 0, p, q, p, q)
        q_guess = estimated_guess2(S, 0, q)
        p_guess = len(S)
        print(f"Estimated for r:{r}, n:{n} p: {p_guess}, Estimated q: {q_guess}")

Estimated for r:3, n:1 p: 3, Estimated q: 1
Estimated for r:3, n:2 p: 10, Estimated q: 3
Estimated for r:3, n:3 p: 36, Estimated q: 10
Estimated for r:3, n:4 p: 136, Estimated q: 36
Estimated for r:3, n:5 p: 528, Estimated q: 136
Estimated for r:3, n:6 p: 2080, Estimated q: 528
Estimated for r:3, n:7 p: 8256, Estimated q: 2080
Estimated for r:4, n:1 p: 4, Estimated q: 1
Estimated for r:4, n:2 p: 18, Estimated q: 4
Estimated for r:4, n:3 p: 86, Estimated q: 18
Estimated for r:4, n:4 p: 422, Estimated q: 86
Estimated for r:4, n:5 p: 2094, Estimated q: 422
Estimated for r:4, n:6 p: 10438, Estimated q: 2094
Estimated for r:4, n:7 p: 52126, Estimated q: 10438
Estimated for r:5, n:1 p: 5, Estimated q: 1
Estimated for r:5, n:2 p: 28, Estimated q: 5
Estimated for r:5, n:3 p: 164, Estimated q: 28
Estimated for r:5, n:4 p: 976, Estimated q: 164
Estimated for r:5, n:5 p: 5840, Estimated q: 976
Estimated for r:5, n:6 p: 35008, Estimated q: 5840
Estimated for r:5, n:7 p: 209984, Estimated q: 35008


In [14]:
for r in tqdm(range(3,10)):
    for p in range(2,8):
        q_p = int((1 + r**p * (r - 2)) / (r - 1))
        q_pm1 = int((1 + (r) ** (p - 1) * (r - 2)) / (r - 1))
        q_pp1 = int((1 + r**(p+1) * (r - 2)) / (r - 1))
        for n in range(1,p):
            for m in range(1,q_p):
                if (m*r**n) % q_p >= q_pm1:
                    if (m*r**(n+1)) % q_pp1 < q_p:
                        print(f"OH NEE TOCH, r: {r}, p: {p}, n: {n}, m: {m}")

100%|██████████| 7/7 [00:27<00:00,  3.90s/it]


In [7]:
import numpy as np

set = []
for i in range(-18,18):
    generator = np.array([[10,-4,2],[2,10,-4],[4,2,10]])
    for j in range(-18,18):
        for k in range(-9,9):
            x = (i*generator[0]+j*generator[1]+k*generator[2])
            set.append(x)

#Make an array of all unique rows in set for which all values are lower than 36 and higher or equal to 0, and print it
unique_set = np.unique(set, axis=0)
filtered_set = unique_set[np.all((unique_set >= 0) & (unique_set < 36), axis=1)]
print(filtered_set, len(filtered_set))


[[ 0  0  0]
 [ 0  0 32]
 [ 0 18 14]
 [ 2 10 28]
 [ 2 28 10]
 [ 4  2 10]
 [ 4 20 24]
 [ 6 12  6]
 [ 6 30 20]
 [ 8  4 20]
 [ 8 22  2]
 [ 8 22 34]
 [10 14 16]
 [10 32 30]
 [12  6 30]
 [12 24 12]
 [14 16 26]
 [14 34  8]
 [16  8  8]
 [16 26 22]
 [18  0 22]
 [18 18  4]
 [20 10 18]
 [20 28  0]
 [20 28 32]
 [22  2  0]
 [22  2 32]
 [22 20 14]
 [24 12 28]
 [24 30 10]
 [26  4 10]
 [26 22 24]
 [28 14  6]
 [28 32 20]
 [30  6 20]
 [30 24  2]
 [30 24 34]
 [32 16 16]
 [32 34 30]
 [34  8 30]
 [34 26 12]] 41
